# 🏥 MedBot Phase 3 - Complete Medical RAG System

## Deep Learning Project - Medical Question Answering

**Developer:** Anamay  
**Repository:** https://github.com/MarcusV210/MedBot/tree/Anamay

---

### 📋 Project Overview

This notebook implements a complete Medical RAG (Retrieval-Augmented Generation) system with:

1. **Baseline Model**: Trained LSTM on medical text (20 epochs with validation)
2. **Medical RAG Model 1**: GPT-3.5 with medical prompting
3. **Medical RAG Model 2**: GPT-4 with advanced medical reasoning

### 🎯 Goals
- Process Harrison's Principles of Internal Medicine (15,000 pages)
- Train baseline model with proper validation
- Achieve 80%+ accuracy on medical Q&A
- Compare all 3 models with comprehensive metrics

### 📊 Dataset
- **Training**: Harrison's medical textbook chunks
- **Evaluation**: 25 medical Q&A pairs from FAQ_Test.csv

## Step 1: Setup and Configuration

**What**: Install dependencies and configure system  
**Why**: Need all libraries for training, RAG, and evaluation  
**How**: Import packages and set hyperparameters

In [ ]:
# Install required packages (run once)
!pip install -q torch sentence-transformers chromadb openai langchain-community pypdf rouge-score scikit-learn matplotlib seaborn tqdm pandas

print("✅ All packages installed!")

In [ ]:
# Import all required libraries
import os, time, warnings, json, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from typing import List, Dict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')

print("✅ All libraries imported successfully!")

In [ ]:
# Configuration
OPENROUTER_API_KEY = "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"

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🎯 Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# Training hyperparameters (OPTIMIZED FOR SPEED)
EMBEDDING_DIM = 128  # Smaller for faster training
HIDDEN_DIM = 256
OUTPUT_DIM = 384
BATCH_SIZE = 128  # Larger batches = faster
LEARNING_RATE = 0.01  # Higher LR = faster convergence
NUM_EPOCHS = 20

# Data processing
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
MAX_PAGES = 500  # Limit pages for faster loading

print(f"\n📊 Configuration:")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Max Pages: {MAX_PAGES}")